# Cleaning our Datasets

In [1]:
import pandas as pd
import string
import re
import numpy as np

import sys
sys.path.append('../src')
from helpers import column_utils, csv_utils, merge_utils

In [3]:
df_draft = csv_utils.csv_read('../data/nfl_draft_data.csv', ';')
df_ras = csv_utils.csv_read('../data/RAS_1987_present.csv', ',')

## To start, we need to perform some cleaning tasks on the columns we will merge on. These include:
- Player Name
- Team Name
- Draft Round


***

### Player Name

In [ ]:
df_draft['name'] = column_utils.column_remove_pattern(
    column_utils.column_strip_whitespace(
        column_utils.column_to_lowercase(df_draft['name'])
    ),
    column_utils.REGEX_NAME_PATTERN,
    ''
)

df_ras['Name'] = column_utils.column_remove_pattern(
    column_utils.column_strip_whitespace(
        column_utils.column_to_lowercase(df_ras['Name'])
    ),
    column_utils.REGEX_NAME_PATTERN,
    ''
)

### Team Name



We want to only include the mascot name in the team name. Our RAS dataset already has this, so we will just strip whitespace and move it to lowercase. Our Draft dataset will require us to do a few other things

- Replace historical team names with their current variant
    - Some team names will be straightforward to deal with even if they are the old team name. These include names like the St. Louis Rams, and the Los Angeles Rams. This is the same franchise, so using just the mascot here is a simple change. 
    - The Washington Commanders however have had other completely different mascots in their past, we will fix this by replacing instances of their old team names (*ex. washington football team*)
    - We can then apply our function to remove the city from the team name and only keep the mascot. We will also apply the same whitespace trim and to lowercase function.

In [5]:
df_draft['team'] = column_utils.column_replace_value(df_draft['team'], column_utils.WASHINGTON_TEAM_NAMES, 'commanders')

df_draft['team'] = column_utils.column_replace_value(df_draft['team'], column_utils.HOUSTON_TEAM_NAMES, 'titans')

In [6]:
df_draft['team'] = column_utils.column_strip_whitespace(
    column_utils.column_to_lowercase(
        column_utils.column_remove_pattern(df_draft['team'], column_utils.REGEX_TEAM_PATTERN, r'\1')
    )
)

df_ras['Draft Team'] = column_utils.column_strip_whitespace(
    column_utils.column_to_lowercase(df_ras['Draft Team'])
)

### Draft Round

In [7]:
df_draft['draft_round'] = column_utils.column_convert_value(df_draft['draft_round'], 'Int64')

***

# Merging Our Datasets

In [10]:
df_merged_draft_base_left = merge_utils.merge_datasets(
    df_draft, df_ras, 
    ['name', 'team', 'draft_round', 'year'], 
    ['Name', 'Draft Team', 'Round', 'Year'], 
    'left', 'm:1')

df_merged_ras_base_right = merge_utils.merge_datasets(
    df_draft, df_ras, 
    ['name', 'team', 'draft_round', 'year'], 
    ['Name', 'Draft Team', 'Round', 'Year'], 
    'right', 'm:1')

***

### Converting to CSV

In [11]:
csv_utils.csv_write(df_merged_draft_base_left, '../data/cleaned/merged_draft_base_left_join_copy.csv')
csv_utils.csv_write(df_merged_ras_base_right, '../data/cleaned/merged_ras_base_right_join_copy.csv')